# Station ID Crosswalk

Builds a durable, human-reviewable mapping from unmatched trip station IDs (`start_station_id` / `end_station_id`) to the current station reference table (`stations.short_name`), so that ID-based joining — already shown to be more reliable than name-based joining in [02_station_id_mapping_check.ipynb](02_station_id_mapping_check.ipynb) — can be extended to cover as many trips as possible.

**Join keys:** as established in notebook 02, `raw_trips.start_station_id` / `end_station_id` correspond to `stations.short_name`, not `stations.station_id` (a UUID from a different ID space with 0 overlap). This notebook uses `short_name` throughout.

**How this notebook works:**
1. Load trips + stations, identify unmatched trip station IDs (same logic as notebook 02), and profile them (first/last appearance, age, row volume, average duration — same logic as notebook 02's profiling section).
2. For each unmatched ID, generate the top-3 closest-matching reference station names by string similarity (same `difflib` approach as notebook 01), as *candidates* — not automatic matches.
3. Maintain a persistent review CSV at `raw_data/station_id_crosswalk.csv`. On first run it's created with all candidates and a blank `chosen_short_name` column for you to fill in by hand. On subsequent runs, your manual choices are preserved and only genuinely new unmatched IDs are appended.
4. Apply whatever has been manually resolved so far to recompute the row-level match rate, so you can see the crosswalk's payoff incrementally as you fill it in.

**You are expected to periodically open the CSV, fill in `chosen_short_name` for `unresolved` rows (or mark them `RETIRED`), save, and re-run the later cells** (no need to re-run the whole notebook) to pick up your edits.

In [7]:
from difflib import SequenceMatcher, get_close_matches
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"
STATIONS_PATH = RAW_DATA / "stations.parquet"
RAW_TRIPS_PATH = RAW_DATA / "raw_trips"
CROSSWALK_PATH = RAW_DATA / "station_id_crosswalk.csv"

print(f"Project root: {ROOT}")
print(f"Stations path: {STATIONS_PATH} (exists={STATIONS_PATH.exists()})")
print(f"Raw trips path: {RAW_TRIPS_PATH} (exists={RAW_TRIPS_PATH.exists()})")
print(f"Crosswalk path: {CROSSWALK_PATH} (exists={CROSSWALK_PATH.exists()})")

Project root: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor
Stations path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\stations.parquet (exists=True)
Raw trips path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\raw_trips (exists=True)
Crosswalk path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\station_id_crosswalk.csv (exists=True)


## Load data

Same schema-override approach as notebook 02: `start_station_id` / `end_station_id` are `int64` in older partitions but become alphanumeric strings from 2021-02 onward, so we open the dataset via `pyarrow.dataset` with an explicit schema forcing both ID columns to `string`.

In [8]:
stations = pd.read_parquet(STATIONS_PATH)

_raw_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive")
_id_string_fields = []
for field in _raw_dataset.schema:
    if field.name in ("start_station_id", "end_station_id"):
        _id_string_fields.append(pa.field(field.name, pa.string()))
    else:
        _id_string_fields.append(field)
_trips_schema = pa.schema(_id_string_fields)

trips_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive", schema=_trips_schema)
trips = trips_dataset.to_table(
    columns=[
        "start_station_id",
        "end_station_id",
        "start_station_name",
        "end_station_name",
        "started_at",
        "ended_at",
        "duration",
        "period",
    ]
).to_pandas()

trips["started_at"] = pd.to_datetime(trips["started_at"], errors="coerce")
trips["ended_at"] = pd.to_datetime(trips["ended_at"], errors="coerce")

station_ids = set(stations["short_name"].dropna().unique())
station_names = set(stations["name"].dropna().unique())

print(f"stations: {stations.shape[0]:,} rows")
print(f"trips:    {trips.shape[0]:,} rows")
print(f"Reference station IDs (stations.short_name): {len(station_ids):,}")

stations: 866 rows
trips:    55,897,151 rows
Reference station IDs (stations.short_name): 866


## Identify and profile unmatched trip station IDs

Same logic as notebook 02: a trip row's `start_station_id`/`end_station_id` is "matched" if it's found in `stations.short_name`. For each unmatched ID, compute row volume (as start/end), an example observed name, first/last appearance, age, and average trip duration — this context helps decide whether an ID is worth resolving manually.

In [9]:
start_id_matched = trips["start_station_id"].isin(station_ids)
end_id_matched = trips["end_station_id"].isin(station_ids)

start_id_counts = trips.loc[~start_id_matched, "start_station_id"].value_counts().rename("as_start")
end_id_counts = trips.loc[~end_id_matched, "end_station_id"].value_counts().rename("as_end")

unmatched_id_impact = pd.concat([start_id_counts, end_id_counts], axis=1).fillna(0).astype(int)
unmatched_id_impact["total_rows"] = unmatched_id_impact["as_start"] + unmatched_id_impact["as_end"]

name_by_start_id = trips.loc[~start_id_matched].groupby("start_station_id")["start_station_name"].first()
name_by_end_id = trips.loc[~end_id_matched].groupby("end_station_id")["end_station_name"].first()
unmatched_id_impact["example_name"] = name_by_start_id.combine_first(name_by_end_id)
unmatched_id_impact = unmatched_id_impact.sort_values("total_rows", ascending=False)

unmatched_ids = unmatched_id_impact.index

start_occurrences = trips.loc[
    trips["start_station_id"].isin(unmatched_ids),
    ["start_station_id", "started_at", "duration"],
].rename(columns={"start_station_id": "station_id", "started_at": "timestamp"})
end_occurrences = trips.loc[
    trips["end_station_id"].isin(unmatched_ids),
    ["end_station_id", "ended_at", "duration"],
].rename(columns={"end_station_id": "station_id", "ended_at": "timestamp"})
occurrences = pd.concat([start_occurrences, end_occurrences], ignore_index=True)

profile = occurrences.groupby("station_id").agg(
    first_appearance=("timestamp", "min"),
    last_appearance=("timestamp", "max"),
    avg_duration_sec=("duration", "mean"),
)
profile["age_days"] = (profile["last_appearance"] - profile["first_appearance"]).dt.total_seconds() / 86400
profile["n_rows"] = unmatched_id_impact["total_rows"]
profile["example_name"] = unmatched_id_impact["example_name"]
profile = profile[
    ["example_name", "first_appearance", "last_appearance", "age_days", "n_rows", "avg_duration_sec"]
].sort_values("n_rows", ascending=False)

print(f"{len(profile):,} unmatched trip station IDs (of {len(unmatched_ids):,} total unique trip IDs)")
profile.head(20)

60 unmatched trip station IDs (of 60 total unique trip IDs)


,example_name,first_appearance,last_appearance,age_days,n_rows,avg_duration_sec
station_id,,,,,,
31200,Massachusetts Ave & Dupont Circle NW,2010-09-20 12:30:50,2025-04-01 09:46:55,5306.886169,1256720,972.745679
31613,Eastern Market Metro / Pennsylvania Ave & 7th St SE,2010-10-06 15:03:26,2024-11-19 09:24:41,5157.764757,808168,852.663755
31248,Smithsonian-National Mall / Jefferson Dr & 12th St SW,2012-04-02 08:49:01,2026-06-08 08:14:24,5179.975961,801402,2550.297562
31288,4th St & Madison Dr NW,2016-06-14 08:10:44,2026-06-05 08:27:54,3643.011921,598826,2155.167836
31246,M St & Pennsylvania Ave NW,2011-12-21 12:01:12,2025-06-11 15:44:49,4921.155289,554805,1198.906237
31614,11th & H St NE,2011-12-12 18:11:40,2021-11-30 06:55:59,3640.530775,198327,764.729241
31642,New Jersey Ave & L St NW,2016-08-02 08:42:47,2025-10-14 09:24:45,3360.029144,151219,849.399165
31008,12th & Army Navy Dr,2010-09-21 09:37:21,2019-01-03 09:23:14,3025.990197,63571,1155.617546
31051,Arlington Blvd & N Queen St,2012-09-05 07:00:42,2025-09-24 07:46:44,4767.031968,53477,999.967493


## Generate fuzzy-match candidates

For each unmatched ID's observed `example_name`, find the top-3 closest `stations.name` values by string similarity (`difflib`, same approach as notebook 01) as candidates for manual review. These are suggestions only — nothing here is applied automatically.

In [10]:
def normalize(name: str) -> str:
    return name.strip().casefold().replace("&", " ").replace("-", " ").strip()


# Map normalized station name -> (short_name, original display name). If multiple
# reference stations normalize to the same name (shouldn't happen, but be defensive),
# keep the first one seen.
norm_to_station = {}
for _, row in stations.dropna(subset=["name", "short_name"]).iterrows():
    key = normalize(row["name"])
    norm_to_station.setdefault(key, (row["short_name"], row["name"]))

norm_station_names = list(norm_to_station.keys())

SIMILARITY_CUTOFF = 0.6
MAX_CANDIDATES = 3

candidate_rows = []
for trip_id, row in profile.iterrows():
    example_name = row["example_name"]
    candidates = []
    if isinstance(example_name, str):
        norm_name = normalize(example_name)
        close = get_close_matches(norm_name, norm_station_names, n=MAX_CANDIDATES, cutoff=SIMILARITY_CUTOFF)
        for candidate_key in close:
            short_name, display_name = norm_to_station[candidate_key]
            similarity = round(SequenceMatcher(None, norm_name, candidate_key).ratio(), 3)
            candidates.append((short_name, display_name, similarity))

    candidate_row = {"trip_station_id": trip_id}
    for i in range(MAX_CANDIDATES):
        if i < len(candidates):
            short_name, display_name, similarity = candidates[i]
        else:
            short_name, display_name, similarity = (None, None, None)
        candidate_row[f"candidate_{i+1}_short_name"] = short_name
        candidate_row[f"candidate_{i+1}_name"] = display_name
        candidate_row[f"candidate_{i+1}_similarity"] = similarity
    candidate_rows.append(candidate_row)

candidates_df = pd.DataFrame(candidate_rows).set_index("trip_station_id")

n_with_candidates = candidates_df["candidate_1_short_name"].notna().sum()
print(f"{n_with_candidates:,} / {len(candidates_df):,} unmatched IDs have at least one fuzzy-match candidate")
candidates_df.head(10)

37 / 60 unmatched IDs have at least one fuzzy-match candidate


,candidate_1_short_name,candidate_1_name,candidate_1_similarity,candidate_2_short_name,candidate_2_name,candidate_2_similarity,candidate_3_short_name,candidate_3_name,candidate_3_similarity
trip_station_id,,,,,,,,,
31200,31657,Massachusetts Ave & 6th St NE,0.708,31395,Massachusetts & Idaho Ave NW,0.625,31647,7th St & Massachusetts Ave NE,0.615
31613,31606,Potomac & Pennsylvania Ave SE,0.625,31701,Branch & Pennsylvania Ave SE,0.608,31269,3rd St & Pennsylvania Ave SE,0.608
31248,31247,Jefferson Dr & 14th St SW,0.615,NaN,NaN,NaN,NaN,NaN,NaN
31288,32348,4th St & Jefferson Dr SW,0.739,32140,18th St & Belmont Rd NW,0.711,31226,34th St & Wisconsin Ave NW,0.708
31246,30200,9th St & Pennsylvania Ave NW,0.926,31237,25th St & Pennsylvania Ave NW,0.909,31252,21st St & Pennsylvania Ave NW,0.909
31614,31611,13th & H St NE,0.929,31672,10th & H St NE,0.929,31661,8th & H St NE,0.889
31642,31509,New Jersey Ave & R St NW,0.958,31655,New Jersey Ave & F St NW,0.958,31828,New Jersey Ave & H St SE,0.875
31008,31542,12th & Varnum St NE,0.632,NaN,NaN,NaN,NaN,NaN,NaN
31051,31141,Arlington Blvd & N Queen St,1.000,32268,Arlington Blvd & Allen St,0.846,31070,Arlington Blvd & N Fillmore St,0.807


## Generate / merge the persistent crosswalk CSV

- **First run** (no CSV yet): write out one row per unmatched ID with all profile + candidate columns, and a blank `chosen_short_name` for you to fill in.
- **Later runs** (CSV exists): keep every existing row exactly as you left it (never overwrite `chosen_short_name` or `status`), refresh the context columns (`n_rows`, `first_appearance`, `last_appearance`, `age_days`, `example_name`, candidates) only for rows that are still unresolved (so a `resolved`/`retired` row's candidates don't keep changing under you), and append any newly-discovered unmatched IDs as new rows.

`status` is derived automatically from `chosen_short_name`:
- blank → `unresolved`
- `RETIRED` (case-insensitive) → `retired`
- any other value → `resolved` (further validated as a real `stations.short_name` when the crosswalk is applied, below)

In [11]:
CANDIDATE_COLS = [
    col for i in range(1, MAX_CANDIDATES + 1)
    for col in (f"candidate_{i}_short_name", f"candidate_{i}_name", f"candidate_{i}_similarity")
]
CONTEXT_COLS = ["example_name", "first_appearance", "last_appearance", "age_days", "n_rows"]
ALL_COLS = ["trip_station_id"] + CONTEXT_COLS + CANDIDATE_COLS + ["chosen_short_name", "status"]


def derive_status(chosen_short_name) -> str:
    if pd.isna(chosen_short_name) or str(chosen_short_name).strip() == "":
        return "unresolved"
    if str(chosen_short_name).strip().upper() == "RETIRED":
        return "retired"
    return "resolved"


# Freshly computed context for every currently-unmatched id.
fresh = profile.join(candidates_df)
fresh.index.name = "trip_station_id"

if CROSSWALK_PATH.exists():
    existing = pd.read_csv(
        CROSSWALK_PATH,
        dtype={"trip_station_id": str, "chosen_short_name": str},
        parse_dates=["first_appearance", "last_appearance"],
    )
    existing = existing.set_index("trip_station_id")

    still_unresolved_ids = existing.index[existing["chosen_short_name"].isna() | (existing["chosen_short_name"].str.strip() == "")]
    new_ids = fresh.index.difference(existing.index)
    refresh_ids = existing.index.intersection(still_unresolved_ids)

    # Refresh context/candidates for still-unresolved rows, and add brand-new rows;
    # never touch rows that already have a human-provided chosen_short_name.
    updated = existing.copy()
    for col in CONTEXT_COLS + CANDIDATE_COLS:
        if col in fresh.columns:
            # Rebuild the column as an object Series to sidestep any dtype mismatch
            # between what's stored in the CSV and freshly computed values, then let
            # pandas re-infer dtype.
            refreshed_values = fresh.loc[fresh.index.intersection(refresh_ids), col]
            col_values = updated[col].astype(object)
            col_values.loc[refresh_ids] = refreshed_values.reindex(refresh_ids).values
            updated[col] = col_values

    new_rows = fresh.loc[new_ids].copy()
    new_rows["chosen_short_name"] = pd.NA
    new_rows["status"] = "unresolved"

    crosswalk = pd.concat([updated, new_rows[updated.columns] if len(new_rows) else new_rows])
    crosswalk["status"] = crosswalk["chosen_short_name"].apply(derive_status)

    print(f"Loaded existing crosswalk: {len(existing):,} rows")
    print(f"  Newly appended (not previously seen as unmatched): {len(new_ids):,}")
    print(f"  Refreshed context for still-unresolved rows: {len(refresh_ids):,}")
else:
    crosswalk = fresh.copy()
    crosswalk["chosen_short_name"] = pd.NA
    crosswalk["status"] = "unresolved"
    print(f"No existing crosswalk found — creating a new one with {len(crosswalk):,} rows")

crosswalk.index.name = "trip_station_id"
crosswalk = crosswalk.reset_index()
crosswalk = crosswalk[ALL_COLS].sort_values("n_rows", ascending=False)
crosswalk.to_csv(CROSSWALK_PATH, index=False)

status_counts = crosswalk["status"].value_counts()
print(f"\nCrosswalk written to: {CROSSWALK_PATH}")
print(status_counts.to_string())

Loaded existing crosswalk: 60 rows
  Newly appended (not previously seen as unmatched): 0
  Refreshed context for still-unresolved rows: 53

Crosswalk written to: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\station_id_crosswalk.csv
status
unresolved    53
resolved       7


## Apply the crosswalk and measure the payoff

Combine (a) trip station IDs that already match `stations.short_name` directly, with (b) `resolved` rows from the crosswalk CSV (validated against the real `stations.short_name` set — a typo'd `chosen_short_name` is reported, not silently trusted), to build a final `id_to_short_name` mapping. Recompute the row-level match rate and compare it against the ID-only baseline (81.33%, from notebook 02) to see how much the manually-resolved crosswalk entries have recovered so far.

In [12]:
resolved_rows = crosswalk[crosswalk["status"] == "resolved"].copy()

# Validate: every chosen_short_name should be a real stations.short_name.
invalid_mask = ~resolved_rows["chosen_short_name"].isin(station_ids)
invalid_rows = resolved_rows.loc[invalid_mask, ["trip_station_id", "chosen_short_name"]]
if len(invalid_rows):
    print(f"WARNING: {len(invalid_rows):,} resolved row(s) have a chosen_short_name not found in stations.short_name:")
    print(invalid_rows.to_string(index=False))
resolved_rows = resolved_rows.loc[~invalid_mask]

# id -> short_name mapping: identity for already-matched ids, crosswalk choice for resolved ones.
id_to_short_name = {sid: sid for sid in station_ids}
id_to_short_name.update(dict(zip(resolved_rows["trip_station_id"], resolved_rows["chosen_short_name"])))

trips["start_resolved"] = trips["start_station_id"].map(id_to_short_name)
trips["end_resolved"] = trips["end_station_id"].map(id_to_short_name)

start_matched_after = trips["start_resolved"].notna()
end_matched_after = trips["end_resolved"].notna()
both_matched_after = start_matched_after & end_matched_after

both_id_match_rate_before = (start_id_matched & end_id_matched).mean()
both_match_rate_after = both_matched_after.mean()

print(f"Resolved crosswalk entries applied: {len(resolved_rows):,} / {len(crosswalk):,} unmatched IDs")
print(f"Retired (confirmed no-match):        {(crosswalk['status'] == 'retired').sum():,}")
print(f"Still unresolved:                    {(crosswalk['status'] == 'unresolved').sum():,}")
print()
print(f"Row-level match rate, ID-only baseline (notebook 02):        {both_id_match_rate_before:.2%}")
print(f"Row-level match rate, after applying crosswalk:              {both_match_rate_after:.2%}")
print(f"Improvement:                                                 +{(both_match_rate_after - both_id_match_rate_before):.2%}")

Resolved crosswalk entries applied: 7 / 60 unmatched IDs
Retired (confirmed no-match):        0
Still unresolved:                    53

Row-level match rate, ID-only baseline (notebook 02):        81.33%
Row-level match rate, after applying crosswalk:              81.45%
Improvement:                                                 +0.12%


## Next steps: how to resolve more IDs

1. Open `raw_data/station_id_crosswalk.csv` (e.g. in Excel or VS Code).
2. For each `unresolved` row, look at `candidate_1_name` / `candidate_2_name` / `candidate_3_name` and their `_similarity` scores, plus `example_name`, `n_rows`, and `age_days` for context (a high `n_rows` + high `age_days` row is worth investigating even if the candidates are a weak match — try looking the station up directly, e.g. by searching its `example_name`).
3. Fill in `chosen_short_name`:
   - Copy a candidate's `short_name` value if it's a correct match.
   - Enter a different `short_name` if you find the right match some other way.
   - Enter `RETIRED` if you've confirmed there's no current equivalent station (e.g. a decommissioned station, a test/placeholder entry like `V1 Warehouse Test Station`).
   - Leave blank to keep it `unresolved` for later.
4. Save the CSV, then re-run **only** the "Generate / merge the persistent crosswalk CSV" and "Apply the crosswalk and measure the payoff" cells above (no need to reload trips/stations or regenerate candidates) to see your changes take effect and the updated match rate.

The crosswalk is sorted by `n_rows` descending, so the rows worth resolving first (biggest impact on the match rate) are already at the top.